# Notebook 3: More than one predictor (solutions)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/centre-for-humanities-computing/introduction-bayesian-statistics/blob/master/notebooks/solutions/03_multiple_predictors_solutions.ipynb)

**Session 1 - Linear regressions** · Block 3

In this notebook you will:

1. explore real data on children's test scores,
2. interpret a **binary** predictor,
3. **center** a predictor so that the intercept means something,
4. fit and interpret a model with **two predictors**, with `statsmodels` and PyMC,
5. (bonus) let the slopes differ between groups with an **interaction**.

> **How to use this notebook.** Run the cells from top to bottom with **Shift + Enter**.
> Cells marked ✏️ ask you to complete or change something. If something breaks,
> use *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter).

In [ ]:
# @title ⚙️ Setup: run this cell first (Shift + Enter)
import importlib.util
import subprocess
import sys
import warnings

PACKAGES = ["pymc", "arviz", "statsmodels"]
missing = [p for p in PACKAGES if importlib.util.find_spec(p) is None]
if missing:
    print("Installing", *missing, "... (about a minute)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import arviz as az
import pymc as pm

# Plot colours
BLUE, ROSE, GREEN, GREY = "#5B7FA6", "#BC6D86", "#5E8C4F", "#595959"
plt.rcParams.update(
    {
        "figure.figsize": (7, 4.5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#EDEDED",
        "font.size": 12,
    }
)
print("Ready!")

## 1. The data

434 children from a US survey, used in *Regression and Other Stories* (Gelman, Hill & Vehtari,
2020, chapter 10):

* `kid_score`: the child's score on a cognitive test at age 3–4,
* `mom_hs`: 1 if the mother finished high school, 0 otherwise,
* `mom_iq`: the mother's IQ score.

This is **observational** data: we can describe and compare, but not prove causes.

In [ ]:
from pathlib import Path

URL = (
    "https://raw.githubusercontent.com/centre-for-humanities-computing/"
    "introduction-bayesian-statistics/master/data/kidiq.csv"
)
local = [
    p for p in (Path("../data/kidiq.csv"), Path("../../data/kidiq.csv")) if p.exists()
]
kid = pd.read_csv(local[0] if local else URL)[["kid_score", "mom_hs", "mom_iq"]]
kid.head()

In [ ]:
kid.describe().round(1)

In [ ]:
colors = {0: "#A6A6A6", 1: "#8BB57C"}
labels = {0: "mother did not finish high school", 1: "mother finished high school"}
fig, ax = plt.subplots()
for hs in (0, 1):
    group = kid[kid["mom_hs"] == hs]
    ax.scatter(
        group["mom_iq"], group["kid_score"], color=colors[hs], s=15, label=labels[hs]
    )
ax.set(xlabel="Mother's IQ", ylabel="Child's test score")
ax.legend()
plt.show()

## 2. A binary predictor

`mom_hs` only takes the values 0 and 1. Let's fit `kid_score ~ mom_hs`.

✏️ **Exercise 2.1.** Fit the model with `smf.ols` and look at the coefficients.

In [ ]:
fit_hs = smf.ols("kid_score ~ mom_hs", data=kid).fit()
fit_hs.params

✏️ **Exercise 2.2.** Now compute the **average score in each group** with
`kid.groupby("mom_hs")["kid_score"].mean()`. How do these two numbers relate to the intercept
and the slope?

In [ ]:
kid.groupby("mom_hs")["kid_score"].mean()

✅ **Answer.** The intercept (77.5) is the average score when `mom_hs = 0`. The slope (11.8) is
the **difference** between the two group averages (89.3 − 77.5). With a 0/1 predictor, the
"line" just connects two group means.

## 3. A continuous predictor and centering

In [ ]:
fit_iq = smf.ols("kid_score ~ mom_iq", data=kid).fit()
fit_iq.params

The slope (0.61) says: comparing mothers whose IQ differs by 1 point, their children's scores
differ by 0.61 points on average. But the **intercept** (25.8) is the predicted score for a
child whose mother has an IQ of **0**, which is meaningless.

**Centering** fixes this: we subtract a meaningful value (IQ 100 = population average) so that
0 now means "average IQ":

In [ ]:
kid["mom_iq_c"] = kid["mom_iq"] - 100
kid[["mom_iq", "mom_iq_c"]].head()

✏️ **Exercise 3.1.** Fit `kid_score ~ mom_iq_c` and compare the coefficients with `fit_iq`.
What changed? What did not?

In [ ]:
fit_iq_c = smf.ols("kid_score ~ mom_iq_c", data=kid).fit()
fit_iq_c.params

✅ **Answer.** The slope is unchanged (0.61). The intercept becomes 86.8: the predicted score
for a child whose mother has an IQ of 100. Same model, easier to read.

## 4. Two predictors

✏️ **Exercise 4.1.** Fit `kid_score ~ mom_hs + mom_iq_c` and store it as `fit_both`.

In [ ]:
fit_both = smf.ols("kid_score ~ mom_hs + mom_iq_c", data=kid).fit()
fit_both.summary().tables[1]

✏️ **Exercise 4.2.** Interpret the three coefficients. Why is the coefficient of `mom_hs`
(≈ 6) about **half** of what it was in section 2 (≈ 11.8)? (Hint: compare the mothers' IQ in
the two groups with `kid.groupby("mom_hs")["mom_iq"].mean()`.)

In [ ]:
kid.groupby("mom_hs")["mom_iq"].mean()

✅ **Answer.**

* **Intercept ≈ 82.1**: predicted score for a child whose mother did not finish high school and
  has an IQ of 100.
* **mom_hs ≈ 6.0**: comparing children whose mothers have the **same IQ**, those whose mothers
  finished high school score about 6 points higher on average.
* **mom_iq_c ≈ 0.56**: comparing children whose mothers have the **same high-school status**,
  each extra IQ point goes with 0.56 more points.

Mothers who finished high school have a higher IQ on average (102 vs 92). On its own, `mom_hs`
also captured part of the IQ difference. Once IQ is in the model, `mom_hs` only captures the
gap **at equal IQ**. A coefficient's meaning depends on what else is in the model.

🎬 **What 'controlling for IQ' does.** Left: each child's score against their mother's IQ, with the IQ slope of the two-predictor model (0.56). Controlling for IQ = setting that slope to 0: every score is adjusted to what it would be at IQ 100 (− 0.56 × (IQ − 100)). The same shift moves the children in the two groups on the right: the gap between the group means shrinks from 11.8 to 6.0 points, exactly the `mom_hs` coefficient, because mothers who finished high school have a higher IQ on average (triangles: 92 vs 102).

![Animation: flattening the IQ slope adjusts every child's score, and the high-school gap shrinks from 11.8 to 6.0](https://raw.githubusercontent.com/centre-for-humanities-computing/introduction-bayesian-statistics/master/animations/renders/03_holding_constant.gif)

The model assumes the **same IQ slope** in both groups, so the two fitted lines are parallel:

In [ ]:
fit_both = smf.ols(
    "kid_score ~ mom_hs + mom_iq_c", data=kid
).fit()  # model of exercise 4.1
xs = np.array([-30, 40])
p = fit_both.params
fig, ax = plt.subplots()
for hs in (0, 1):
    group = kid[kid["mom_hs"] == hs]
    ax.scatter(
        group["mom_iq_c"],
        group["kid_score"],
        color=colors[hs],
        s=12,
        alpha=0.6,
        label=labels[hs],
    )
    ax.plot(
        xs,
        p["Intercept"] + p["mom_hs"] * hs + p["mom_iq_c"] * xs,
        color=["#404040", "#5E8C4F"][hs],
        lw=3,
    )
ax.set(xlabel="mom_iq_c (mother's IQ − 100)", ylabel="Child's test score")
ax.legend()
plt.show()

## 5. The same model in PyMC

Each prior describes what is plausible before seeing the data:

* `intercept ~ Normal(80, 20)`: average test scores are somewhere around 40 to 120,
* `b_hs ~ Normal(0, 10)`: a high-school gap of more than ±20 points would be surprising,
* `b_iq ~ Normal(0, 1)`: one IQ point is unlikely to matter more than ±2 points,
* `sigma ~ HalfNormal(20)`: the scatter around the line is positive, at most a few tens.

In [ ]:
with pm.Model() as kid_model:
    intercept = pm.Normal("intercept", mu=80, sigma=20)
    b_hs = pm.Normal("b_hs", mu=0, sigma=10)
    b_iq = pm.Normal("b_iq", mu=0, sigma=1)
    sigma = pm.HalfNormal("sigma", sigma=20)

    mu = intercept + b_hs * kid["mom_hs"] + b_iq * kid["mom_iq_c"]
    pm.Normal("kid_score", mu=mu, sigma=sigma, observed=kid["kid_score"])

    idata = pm.sample(draws=1000, tune=1000, chains=2, random_seed=1)

az.summary(idata, var_names=["intercept", "b_hs", "b_iq", "sigma"])

✏️ **Exercise 5.1.** Build a small table comparing, for `mom_hs`, the `statsmodels` estimate
and 95% confidence interval with the PyMC posterior mean and 95% credible interval.
(Hint: `fit_both.conf_int().loc["mom_hs"]` and `np.quantile(draws, [0.025, 0.975])`.)

In [ ]:
b_hs_draws = idata.posterior["b_hs"].values.ravel()
ci = fit_both.conf_int().loc["mom_hs"]
cr = np.quantile(b_hs_draws, [0.025, 0.975])
pd.DataFrame(
    {
        "statsmodels": [fit_both.params["mom_hs"], ci.iloc[0], ci.iloc[1]],
        "PyMC": [b_hs_draws.mean(), cr[0], cr[1]],
    },
    index=["estimate", "95% low", "95% high"],
).round(2)

✅ **Answer.** Both give a gap of about 6 points with an interval of roughly 1.5 to 10. With 434
children and weak priors, the two approaches agree; the PyMC version additionally lets us say,
for instance, that P(b_hs > 0 | data) is about 0.99:

In [ ]:
(b_hs_draws > 0).mean()

## 6. Bonus: an interaction

What if the IQ slope is **different** for the two groups? In a formula, `mom_hs * mom_iq_c`
adds both predictors **and** their product (the interaction).

✏️ **Exercise 6.1.** Fit `kid_score ~ mom_hs * mom_iq_c` and compute the IQ slope in each group:

* `mom_hs = 0`: the coefficient of `mom_iq_c`,
* `mom_hs = 1`: the coefficient of `mom_iq_c` + the coefficient of `mom_hs:mom_iq_c`.

In [ ]:
fit_int = smf.ols("kid_score ~ mom_hs * mom_iq_c", data=kid).fit()
p = fit_int.params
print(p.round(2))
print("IQ slope, mother without high school:", round(p["mom_iq_c"], 2))
print(
    "IQ slope, mother with high school:   ",
    round(p["mom_iq_c"] + p["mom_hs:mom_iq_c"], 2),
)

fig, ax = plt.subplots()
for hs in (0, 1):
    group = kid[kid["mom_hs"] == hs]
    ax.scatter(
        group["mom_iq_c"],
        group["kid_score"],
        color=colors[hs],
        s=12,
        alpha=0.6,
        label=labels[hs],
    )
    slope = p["mom_iq_c"] + p["mom_hs:mom_iq_c"] * hs
    ax.plot(
        xs,
        p["Intercept"] + p["mom_hs"] * hs + slope * xs,
        color=["#404040", "#5E8C4F"][hs],
        lw=3,
    )
ax.set(xlabel="mom_iq_c (mother's IQ − 100)", ylabel="Child's test score")
ax.legend()
plt.show()

✅ **Answer.** The IQ slope is about 0.97 when the mother did not finish high school and about
0.48 when she did: the lines are no longer parallel. Interpreting interactions takes care; this
is a topic for the next session.

## Summary

* With a **0/1 predictor**, the slope is a difference between two group means.
* **Centering** a predictor changes the intercept (now meaningful), not the slope.
* In a model with two predictors, each coefficient compares units that are **equal on the other
  predictor**, so coefficients change when predictors are added.
* `statsmodels` and PyMC give nearly the same answers here; PyMC makes every assumption,
  priors included, explicit.

🎯 Time for the **quiz**!